# 02 — IVIVE: Microsomal Stability → Hepatic Clearance

Demonstrates the In Vitro to In Vivo Extrapolation workflow using the well-stirred model.

**Workflow:**
1. Measure microsomal half-life (t½) from substrate depletion assay
2. Calculate in vitro intrinsic clearance (CLint)
3. Scale to whole-liver CLint using MPPGL and liver weight
4. Apply the well-stirred model to predict hepatic clearance (CLh)

In [ ]:
import sys
sys.path.insert(0, '..')

from doseprojection.ivive import (
    calc_clint_invitro, scale_clint,
    predict_hepatic_clearance, predict_extraction_ratio,
    predict_fh, ivive_workflow
)
from doseprojection.constants import MPPGL, LIVER_WEIGHT, HEPATIC_BLOOD_FLOW

## Step-by-Step IVIVE

Example: compound with microsomal t½ = 30 min in human liver microsomes, fu = 0.10

In [ ]:
# Step 1: In vitro CLint from substrate depletion
clint_iv = calc_clint_invitro(t_half_min=30, volume_ul=1000, protein_mg=0.5)
print(f"In vitro CLint: {clint_iv:.1f} uL/min/mg protein")

# Step 2: Scale to whole liver
clint_scaled = scale_clint(clint_iv, species="human")
print(f"Scaled CLint: {clint_scaled:.1f} mL/min/kg")
print(f"  MPPGL = {MPPGL['human']} mg/g, Liver = {LIVER_WEIGHT['human']} g/kg")

# Step 3: Well-stirred model
fu = 0.10
cl_h = predict_hepatic_clearance(clint_scaled, fu, species="human")
eh = predict_extraction_ratio(clint_scaled, fu, species="human")
fh = predict_fh(eh)

print(f"\nPredicted hepatic clearance: {cl_h:.2f} mL/min/kg")
print(f"Extraction ratio: {eh:.3f}")
print(f"Hepatic bioavailability (Fh): {fh:.3f}")
print(f"QH (human): {HEPATIC_BLOOD_FLOW['human']} mL/min/kg")

## Convenience: Full IVIVE Workflow

In [ ]:
result = ivive_workflow(t_half_min=30, fu=0.10, species="human")
for key, val in result.items():
    print(f"{key}: {val:.4f}")

## Compare Across Species

Same compound tested in rat and human microsomes.

In [ ]:
import pandas as pd

species_data = {
    'human': {'t_half': 30, 'fu': 0.10},
    'rat':   {'t_half': 15, 'fu': 0.15},
}

rows = []
for sp, params in species_data.items():
    r = ivive_workflow(params['t_half'], params['fu'], sp)
    r['species'] = sp
    rows.append(r)

df = pd.DataFrame(rows).set_index('species')
df